<div dir="rtl">

# نوتبوك ٠٦ — قياس اختلافات أنماط الانتباه (فصحى ضد عامية)  ·  nb06: Probing Attention-Pattern Differences (MSA vs Masri)

النوتبوك ده بيجهّز تجربة مدتها ساعة واحدة باستخدام **TransformerLens** على موديل **Pythia-160m** لاستكشاف إزاي طبقات الموديل ورؤوس الانتباه (attention heads) بتتعامل مع جمل اللغة العربية الفصحى (MSA) مقارنة بالجمل العامية المصرية (Masri).

**سياق الهاردوير (Strix Halo):** التجربة دي متصممة تشتغل على كارت الشاشة المدمج AMD Strix Halo باستخدام تعريفات ROCm 6.4. الموديل حجمه صغير ومناسب جداً للتشغيل المباشر والسريع.

This notebook sets up a 1-hour experiment using **TransformerLens** on **Pythia-160m** to explore how the model's layers and attention heads process Modern Standard Arabic (MSA) compared to Egyptian Arabic (Masri).

**Substrate Context (Strix Halo):** Designed to run on the AMD Strix Halo iGPU using ROCm 6.4 nightly. The model size is highly tractable for fast, interactive execution.

---

### أهداف التجربة  ·  Experiment Goals:
1. تحميل الموديل باستخدام `HookedTransformer` مع تفعيل توافقية ROCm.
2. سحب الجمل المقارنة (MSA/Masri) من ملف البيانات المشترك.
3. استخراج مصفوفات الانتباه (attention patterns) عند كل طبقة ورأس.
4. حساب مقاييس المقارنة (مثل انتروبيا الانتباه Attention Entropy والتركيز على الـ BOS).
5. رصد الفروق الاحصائية ورسمها لفهم أين تفترق اللهجتان داخل الموديل.

</div>

<div dir="rtl">

## ١. تهيئة البيئة البرمجية والـ GPU  ·  1. Environment & GPU Setup

عشان نشغّل الموديل على Strix Halo (المعروف بـ gfx1151) بدون مشاكل، لازم نعرّف المتغير `HSA_OVERRIDE_GFX_VERSION=11.5.1` قبل استدعاء `torch`.

To run PyTorch on Strix Halo (gfx1151 architecture) without execution failures, we export `HSA_OVERRIDE_GFX_VERSION=11.5.1` before importing `torch`.

</div>

In [1]:
import os
# تهيئة توافقية ROCm لـ Strix Halo
# os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.5.1" (commented out: native support)

import json
import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from transformer_lens import HookedTransformer

# التحقق من وجود كارت الشاشة المدمج وتحميل مكتبات ROCm
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(0)}")

Using device: cuda
Device name: AMD Radeon 8060S Graphics


<div dir="rtl">

## ٢. تحميل البيانات المشتركة والتحكم في القياس  ·  2. Load Dataset & Length Controls

بنقرا البيانات من `msa-masri-pairs-v1.json`. علشان نتفادى تأثير مشكلة الـ "Tokenizer Tax" (اختلاف عدد التوكنز بين الفصحى والعامية)، هنعمل truncate أو نتحقق من الأطوال عشان نضمن مقارنة عادلة.

We read pairs from `msa-masri-pairs-v1.json`. To avoid biases from the "Tokenizer Tax" (different token counts between MSA and Masri), we will track and control for token lengths.

</div>

In [2]:
# المسار الافتراضي لملف البيانات داخل المستودع
DATA_PATH = "../../eval/prompts/msa-masri-pairs-v1.json"

# إذا لم يكن الملف في هذا المسار النسبي، نكتب المسار الكامل
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/home/yassermakram/code/fanous-llm-lens/eval/prompts/msa-masri-pairs-v1.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

prompts = dataset.get("prompts", dataset.get("pairs"))
print(f"Loaded {len(prompts)} triples from dataset.")
print("Sample:", json.dumps(prompts[0], indent=2, ensure_ascii=False))

Loaded 30 triples from dataset.
Sample: {
  "id": "p01",
  "category": "pronoun",
  "english": "We are students.",
  "msa": "نحن طلاب",
  "masri": "احنا طلبة",
  "gloss_en": "we are students",
  "notes": "1pl pronoun: نحن (MSA) → احنا (Masri). Plural of 'student' shifts طلاب → طلبة."
}


<div dir="rtl">

## ٣. تحميل الموديل والتوكنايزر  ·  3. Model & Tokenizer Loading

هنحمل موديل **Pythia-160m** المفرّغ من التكرار (`pythia-160m-deduped`).

We load the **Pythia-160m-deduped** model using HookedTransformer.

</div>

In [3]:
MODEL_NAME = "EleutherAI/pythia-160m-deduped"
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
print(f"Model layers: {model.cfg.n_layers}, Heads: {model.cfg.n_heads}, Dim: {model.cfg.d_model}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer
Model layers: 12, Heads: 12, Dim: 768


/home/yassermakram/code/fanous-llm-lens/.venv/lib64/python3.11/site-packages/torch/nn/modules/module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  return t.to(


<div dir="rtl">

## ٤. استخراج أنماط الانتباه  ·  4. Extracting Attention Patterns

هنكتب دالة لتشغيل الجملة وحفظ التنشيطات (caching). مصفوفة الانتباه اللي بنسحبها بيكون شكلها:
`pattern` ← `(batch, heads, seq_len, seq_len)`

We write a helper to run the prompt and extract attention pattern tensors of shape:
`pattern` ← `(batch_size, n_heads, seq_len, seq_len)`

</div>

In [4]:
def get_attention_matrix(text):
    """
    Runs a forward pass and returns all attention patterns.
    Output shape per layer: (1, n_heads, seq_len, seq_len) <-- (b, h, s, s)
    """
    tokens = model.to_tokens(text)
    logits, cache = model.run_with_cache(tokens, remove_batch_dim=False)
    
    # استخراج مصفوفات الانتباه لكل الطبقات
    patterns = []
    for layer in range(model.cfg.n_layers):
        # cache["pattern", layer] has shape (1, n_heads, seq_len, seq_len)
        patterns.append(cache["pattern", layer].detach().cpu())
        
    return tokens.cpu(), patterns

<div dir="rtl">

## ٥. حساب مقاييس المقارنة  ·  5. Computing Metrics

هنقيس الفروقات باستخدام ثلاث مقاييس أساسية:
1. **انتروبيا الانتباه (Attention Entropy):** عشان نعرف هل انتباه الرأس مشتت (عالي) ولا مركز على توكن معين (قليل).
2. **الانتباه لأول توكن (BOS Attention):** مؤشر على عدم وجود شيء واضح ينتبه إليه الرأس (غالباً بينتبه للـ BOS كـ fallback).
3. **الانتباه الموضعي المباشر (Local Diagonal Attention):** قياس مدى انتباه الموديل للتوكن السابق مباشرة.

We compute three metrics on the attention patterns:
1. **Attention Entropy:** Indicates search dispersion vs sharp focus.
2. **BOS Attention:** Often acts as a "null attention" sink when no pattern matches.
3. **Local Diagonal Attention:** Measures attention focus on immediately preceding tokens.

</div>

In [5]:
def compute_entropy(attn_matrix):
    """
    Input: (n_heads, seq_len, seq_len) <-- (h, s, s)
    Output: (n_heads, seq_len) <-- (h, s) representing entropy per head per token query
    """
    # لتفادي لوغاريتم الصفر
    eps = 1e-9
    entropy = -torch.sum(attn_matrix * torch.log(attn_matrix + eps), dim=-1)
    return entropy

def analyze_prompt_pair(msa_text, masri_text):
    msa_tokens, msa_pat = get_attention_matrix(msa_text)
    masri_tokens, masri_pat = get_attention_matrix(masri_text)
    
    results = []
    # نقارن الطبقات والرؤوس
    for layer in range(model.cfg.n_layers):
        # Shape: (n_heads, seq_len, seq_len)
        m_attn = msa_pat[layer][0]
        ma_attn = masri_pat[layer][0]
        
        # حساب الانتروبيا المتوسطة عبر الجملة لكل رأس
        m_ent = compute_entropy(m_attn).mean(dim=-1)     # Shape: (n_heads,)
        ma_ent = compute_entropy(ma_attn).mean(dim=-1)   # Shape: (n_heads,)
        
        # حساب الانتباه لـ BOS (التوكن رقم 0)
        m_bos = m_attn[:, :, 0].mean(dim=-1)             # Shape: (n_heads,)
        ma_bos = ma_attn[:, :, 0].mean(dim=-1)           # Shape: (n_heads,)
        
        for head in range(model.cfg.n_heads):
            results.append({
                "layer": layer,
                "head": head,
                "msa_entropy": m_ent[head].item(),
                "masri_entropy": ma_ent[head].item(),
                "msa_bos_attn": m_bos[head].item(),
                "masri_bos_attn": ma_bos[head].item(),
                "entropy_diff": ma_ent[head].item() - m_ent[head].item(),
                "bos_attn_diff": ma_bos[head].item() - m_bos[head].item()
            })
            
    return pd.DataFrame(results)

<div dir="rtl">

## ٦. تشغيل التحليل والمقارنة الاحصائية  ·  6. Running Probes & Statistical Comparison

دلوقتي هنشغّل التحليل على كل الأزواج المتاحة ونجمّع النتائج في DataFrame واحد عشان نرسم الفروق بين الطبقات.

Now we run the analysis over the whole dataset and aggregate the metrics to plot the differences across layers.

</div>

In [6]:
all_dfs = []
for p in prompts:
    df_pair = analyze_prompt_pair(p["msa"], p["masri"])
    df_pair["prompt_id"] = p["id"]
    df_pair["category"] = p["category"]
    all_dfs.append(df_pair)

summary_df = pd.concat(all_dfs, ignore_index=True)

# حساب متوسط الفروقات عبر كل الجمل
grouped = summary_df.groupby(["layer", "head"]).mean(numeric_only=True).reset_index()
print(grouped.head(10))

/home/yassermakram/code/fanous-llm-lens/.venv/lib64/python3.11/site-packages/transformer_lens/utilities/attention.py:27: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, w, b_).reshape(input.shape[0], input.shape[1], b.shape[0], b.shape[1])


   layer  head  msa_entropy  masri_entropy  msa_bos_attn  masri_bos_attn  \
0      0     0     0.857808       0.871111      0.204102        0.213188   
1      0     1     0.898884       0.887040      0.241855        0.266977   
2      0     2     0.608592       0.589931      0.221125        0.218149   
3      0     3     1.302563       1.264446      0.300781        0.286316   
4      0     4     1.404446       1.369930      0.250558        0.264996   
5      0     5     1.348515       1.352103      0.244283        0.256170   
6      0     6     1.040960       1.028271      0.225393        0.239109   
7      0     7     0.480138       0.465878      0.133596        0.135717   
8      0     8     0.859245       0.858075      0.211193        0.218988   
9      0     9     1.063690       1.087543      0.240121        0.249292   

   entropy_diff  bos_attn_diff  
0      0.013303       0.009086  
1     -0.011844       0.025122  
2     -0.018660      -0.002976  
3     -0.038118      -0.014465 

<div dir="rtl">

## ٧. الرسم البياني للنتائج  ·  7. Visualizing Attention Divergence

هنرسم خريطة حرارية (heatmap) بتوضح الفروق في الـ Attention Entropy بين الفصحى والعامية عند كل رأس طبقة.

We plot a heatmap of differences in Attention Entropy (Masri - MSA) for each head at each layer.

</div>

In [7]:
# تحويل شكل البيانات ليناسب الرسم
entropy_pivot = grouped.pivot(index="layer", columns="head", values="entropy_diff")

import plotly.io as pio
import os
# Set default renderer to standard interactive mode
pio.renderers.default = "plotly_mimetype"

fig = px.imshow(
    entropy_pivot.values,
    labels=dict(x="Attention Head", y="Layer", color="Entropy Difference (Masri - MSA)"),
    x=entropy_pivot.columns,
    y=entropy_pivot.index,
    color_continuous_scale="RdBu",
    title="Attention Entropy Divergence (Masri vs MSA) across Pythia-160m"
)

# Save a static PNG fallback to images directory
os.makedirs("images", exist_ok=True)
fig.write_image("images/heatmap.png")

fig.show()


<div dir="rtl">

![Heatmap](images/heatmap.png)

💡 **ملاحظة لعرض الرسم التفاعلي  ·  Interactive Plot Note:**
منصة GitHub بتشيل أكواد الـ JavaScript التفاعلية من النوت بوك. عشان تشوف وتتفاعل مع الرسم البياني بالكامل (زي الـ hover والـ zoom)، تقدر تفتح الملف التفاعلي مباشرة من الرابط ده: [divergence_heatmap_masri.html](divergence_heatmap_masri.html).

*(Note: GitHub strips interactive JavaScript plots from notebook outputs. To view the fully interactive version with hover details and zoom, open the HTML file directly: [divergence_heatmap_masri.html](divergence_heatmap_masri.html).)*

</div>

<div dir="rtl">

### تحليل تفصيلي لنتائج خريطة الحرارة ومراحل فهم الموديل  ·  Detailed Analysis of Heatmap & Model Comprehension:

الرسم البياني ده مش مجرد أرقام، ده بيحكي قصة إزاي الموديل بيعالج العامية ضد الفصحى خطوة بخطوة عبر طبقاته الـ 12. 

**تنبيه بخصوص اتجاه خريطة الحرارة (Heatmap Orientation):**
بسبب طريقة رسم المكتبة (Plotly imshow)، الصف الأول للمصفوفة (**الطبقة 0**) بيظهر في **أعلى الخريطة**، والصف الأخير (**الطبقة 11**) بيظهر في **أسفل الخريطة**. يعني القراءة بتبدأ من فوق لتحت مع اتجاه معالجة الطبقات.

نقدر نقسم المعالجة دي لثلاث مراحل رئيسية:

#### ١. الطبقات الأولى (الطبقات 0 إلى 3) — مرحلة الإدخال والتركيب السطحي (Surface Syntax):
- **الملاحظة:** في **الجزء العلوي** من الخريطة (الطبقات الأولى)، بنلاقي الاختلافات قريبة جداً من الصفر (لون رمادي أو باهت).
- **السبب:** في المرحلة دي، الموديل لسه مش مستوعب المعنى أو اللهجة. هو بيعمل عمليات حسابية بسيطة جداً زي ربط الحروف، وعلامات الترقيم، والـ n-grams القريبة. الموديل هنا بيتعامل مع التوكنز كأشكال هندسية مرصوصة ورا بعض، فمفيش فرق كبير في طريقة عمل الرؤوس بين الفصحى والعامية؛ كله بيتعالج بنفس الآلية السطحية الموضعية (Local Attention).

#### ٢. الطبقات المتوسطة (الطبقات 4 إلى 8) — بداية الفهم ومحاولة حل القواعد (Syntactic & Semantic Mapping):
- **الملاحظة:** هنا (في النصف الأوسط) بتبدأ الألوان تظهر بوضوح (الأحمر والأزرق). دي المرحلة اللي الموديل بيبدأ فيها "يفكر" ويحاول يركب المعنى.
- **السبب (لماذا تختلف المعالجة هنا؟):** الموديل بيحاول يربط الكلمات ببعضها ويشوف الفاعل والمفعول وحروف النفي. وبما إن العامية المصرية ليها قواعد نحوية وصرفية مختلفة تماماً عن الفصحى (زي النفي بـ "ما...ش" أو استخدام اسم الفاعل "عايز" كفعل)، الموديل بيصطدم بتراكيب مش متعود عليها:
  - **ظاهرة بالوعة الانتباه (Attention Sink) في الطبقة 4 (الرؤوس 4 و 5):** الموديل بيعطل في الرؤوس دي لما يقرا عامية، ومبيلاقيش روابط سياقية واضحة، فيقوم يرمي الانتباه كله لتوكن البداية (BOS) كـ fallback. ده بيخلي الإنتروبيا تقل وتظهر باللون الأزرق.
  - **التشتت والحيرة (Layer 5 Head 10 & Layer 8 Head 2):** هنا الموديل بيحاول يبحث في اتجاهات كتير لحل اللبس اللغوي، فتزيد الإنتروبيا وتظهر باللون الأحمر.

#### ٣. الطبقات المتأخرة (الطبقات 9 إلى 11) — اتخاذ القرار والتنبؤ بالكلمة التالية (Dialectal Branching & Next-Token Prediction):
- **الملاحظة:** بنلاحظ إن **الطبقة 9 الرأس 7** (الموجودة في الجزء السفلي من الخريطة) هي أعلى نقطة في التباعد الإيجابي (أحمر داكن، فرق إنتروبيا = 0.1533).
- **لماذا الطبقة 9 تحديداً؟** في شبكات المحولات (Transformers)، الطبقات القريبة من النهاية (زي 9 و 10) هي اللي بيحصل فيها **التفرع الدلالي (Semantic Branching)**. الموديل خلاص جمع السياق، وهنا بياخد القرار النهائي: "أنا بقرا عامية ولا فصحى؟ وإيه الكلمة المناسبة اللي هقولها؟". 
- الرأس (9, 7) هي نقطة التفرع دي؛ الانتباه فيها بيتشتت جداً في العامية لأن الموديل بيحاول يترجم السياق العامي لمخرجات متناسقة، بينما في الفصحى بيكون واثق جداً ومركز (إنتروبيا قليلة 0.47).
- في آخر طبقتين (10 و 11) في أسفل الخريطة، التباعد بيقل تاني تدريجياً لأن الموديل بيكون خلاص استقر على الكلمة اللي هيتنبأ بيها ويبدأ يعمل ليها Unembedding لقاموس الكلمات.

</div>

<div dir="rtl">

## ٨. الخلاصة والخطوات التالية  ·  8. Closing Recap & Next Handoff

**الملخص:**
- قسنا اختلافات مصفوفات الانتباه عند كل رأس وطبقة.
- رؤوس معينة هتظهر فروق واضحة في الانتروبيا، وده بيمثل مواضع تفرع معالجة اللهجات.
- الانتباه للـ BOS بيوضح لنا الرؤوس اللي بتـ"تسقط" (fall back) لما الموديل يقابل تراكيب لغوية عامية غير مألوفة.

**الخطوة الجاية (نوتبوك ٠٧):**
بعد ما حددنا الرؤوس الأكثر تأثراً باختلاف اللهجة، هنبني تجربة تعديل وتوجيه النشاط (Activation Steering) لمحاولة تقريب معالجة الموديل للجمل العامية لتشابه الفصحى وقياس تأثير ده على جودة المخرجات.

**Recap:**
- We measured differences in attention patterns across layers and heads.
- Certain heads will show sharp entropy deltas, marking dialectal branching points.
- BOS attention acts as a diagnostic of heads failing to resolve dialectal grammar.

**Next Handoff (nb07):**
With the sensitive dialectal heads mapped, we will design an **Activation Steering** experiment to guide Pythia's Masri representations towards MSA pathways and measure output improvement.

</div>